# YC cluster-hybrid — load and predict (8 inputs)

This notebook uses the **saved hybrid bundle** in `artifacts/hybrid_cluster_bundle.joblib` (from `02_yc_hybrid_ensemble.ipynb`) and [`yc_hybrid_inference.py`](yc_hybrid_inference.py).

**Inputs:** Same eight fields as `07b_ensemble_prediction_pipeline.ipynb` (block, street, town, flat type, area, storey range, lease start year, sale month).

**Feature construction:** The YC model expects **77 columns** (see `artifacts/hybrid_cluster_feature_columns.json`), not the 103-feature main pipeline. The helper **looks up** the address in `YC_data/hdb_feature_table_20260406.csv` for POI/market fields, then **overrides** unit-level fields from your inputs. If no row matches, it uses **town-level medians** (see `imputation_note` in the result).

**Limitations:** No SHAP/LIME from 07b apply here. Predictions without a good address match are less reliable.

**Optional actual price:** Set `actual_price` in the prediction cell (transacted, asking, or appraisal) to print an **evaluation table** (MAE, RMSE, MAPE, bias %). For a single listing, MAE and RMSE both equal the absolute error.


In [17]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import importlib
from pathlib import Path
import os

# Run from exploration-on-yc-data/ or repo root
if not Path("yc_hybrid_inference.py").exists() and Path("exploration-on-yc-data/yc_hybrid_inference.py").exists():
    os.chdir("exploration-on-yc-data")

# Reload so Jupyter picks up edits to yc_hybrid_inference.py (avoids stale kernel cache)
import yc_hybrid_inference
importlib.reload(yc_hybrid_inference)

from yc_hybrid_inference import (
    load_bundle,
    predict_from_user_input,
    build_yc_hybrid_vector,
    predict_price,
    predict_bulk_listings,
    bulk_evaluation_summary,
    per_listing_evaluation_table,
    USER_INPUT_KEYS,
)

bundle = load_bundle()
print("Loaded bundle:", bundle["n_clusters"], "clusters")


Loaded bundle: 4 clusters


## Edit your listing below


In [18]:
import pandas as pd
import numpy as np

# Known transacted / asking / appraisal price (SGD), or None to skip evaluation
actual_price = 979_000

user_input = {
    "block": "18C",
    "street_name": "CIRCUIT ROAD",
    "town": "GEYLANG",
    "flat_type": "4 ROOM",
    "floor_area_sqm": 93,
    "storey_range": "07 TO 09",
    "lease_commence_date": 2015,
    "sale_month": "2026-04",
}

result = predict_from_user_input(**user_input, bundle=bundle)
pred = float(result["predicted_resale_price"])
print("Predicted resale price (SGD):", round(pred, 2))
print("Lookup matched:", result["lookup_matched"], "|", result["matched_address_key"])
if result["imputation_note"]:
    print("Note:", result["imputation_note"])

# Full X: one value per feature (YC lookup + overrides), same order as the model
X_series = pd.Series(result["vector"].ravel(), index=bundle["feature_columns"])
print("\n--- Full X (after lookup + overrides) ---")
print(X_series.to_string())

if actual_price is not None:
    y_t = np.array([actual_price], dtype=float)
    y_p = np.array([pred], dtype=float)
    err = y_p - y_t
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    mape = float(np.mean(np.abs(err / y_t)) * 100)
    bias_pct = float((pred - actual_price) / actual_price * 100)
    eval_df = pd.DataFrame(
        {
            "metric": [
                "actual (SGD)",
                "predicted (SGD)",
                "MAE (SGD)",
                "RMSE (SGD)",
                "MAPE (%)",
                "bias (% pred−actual)",
            ],
            "value": [actual_price, pred, mae, rmse, mape, bias_pct],
        }
    )
    # Human-readable numbers (no scientific notation)
    eval_df["value_display"] = [
        f"{actual_price:,.0f}",
        f"{pred:,.2f}",
        f"{mae:,.2f}",
        f"{rmse:,.2f}",
        f"{mape:.4f}",
        f"{bias_pct:.4f}",
    ]
    print("\n--- Evaluation vs actual_price ---")
    print(eval_df[["metric", "value_display"]].rename(columns={"value_display": "value"}).to_string(index=False))
else:
    print("\n(Set actual_price to a number to print the evaluation table.)")


Predicted resale price (SGD): 1013782.17
Lookup matched: True | 18C CIRCUIT RD

--- Full X (after lookup + overrides) ---
transaction_year                       2026.000000
level_mid                                 8.000000
lease_remaining_years                    88.000000
floor_area_sqm                           93.000000
room_count                                4.000000
dist_to_mrt_m                          2275.265112
orientation_score                        -1.000000
dist_to_highway_m                       155.330775
dist_to_foodcourt_m                     512.885493
dist_to_nearest_mall_m                 1312.023163
mall_count_3km                           10.000000
mall_weighted_access_3km                  5.159633
dist_to_nearest_school_m                787.159908
school_count_1km                          1.000000
primary_school_quality_1km_weighted      14.134463
primary_school_top_quality_1km           14.134463
primary_school_count_1km                  1.000000
trans_sold_

## Bulk listings

Pass a **list of dicts**; each must include the eight `USER_INPUT_KEYS` fields. Optional: `listing_id`, `actual_price` (adds error columns). Below: sample rows shaped like PropertyGuru exports (asking price as `actual_price`, `sqft` converted to m²). Storey range is a placeholder where the listing omits it. `per_listing_evaluation_table(bulk_df)` prints one evaluation row per listing; `bulk_evaluation_summary` aggregates.


In [19]:
import pandas as pd

def sqft_to_sqm(sqft: float) -> float:
    return round(sqft * 0.09290304, 2)

# Match listing month when scrape says Apr 2026
SALE_MONTH = "2026-04"
# PropertyGuru often omits storey — neutral mid-band placeholder
STOREY = "07 TO 09"

# Sample bulk inputs (PropertyGuru-style): listing_id, block, street, town, flat_type,
# floor_area from sqft, lease year from "Built:", sale_month, actual_price = asking price
listings = [
    {"listing_id": "pg-101-jurong-east-s13", "block": "101", "street_name": "JURONG EAST STREET 13", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(732), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 530_000},
    {"listing_id": "pg-244-jurong-east-s24", "block": "244", "street_name": "JURONG EAST STREET 24", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(764), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 380_000},
    {"listing_id": "pg-93-paya-lebar", "block": "93", "street_name": "PAYA LEBAR WAY", "town": "GEYLANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(624), "storey_range": STOREY, "lease_commence_date": 1972, "sale_month": SALE_MONTH, "actual_price": 300_000},
    {"listing_id": "pg-241-jurong-east-s24", "block": "241", "street_name": "JURONG EAST STREET 24", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(732), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 468_000},
    {"listing_id": "pg-311c-clementi-av4", "block": "311C", "street_name": "CLEMENTI AVENUE 4", "town": "CLEMENTI", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(646), "storey_range": STOREY, "lease_commence_date": 2015, "sale_month": SALE_MONTH, "actual_price": 675_000},
    {"listing_id": "pg-211-jurong-east-s21", "block": "211", "street_name": "JURONG EAST STREET 21", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 400_000},
    {"listing_id": "pg-131-cashew", "block": "131", "street_name": "CASHEW ROAD", "town": "BUKIT PANJANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(786), "storey_range": STOREY, "lease_commence_date": 1987, "sale_month": SALE_MONTH, "actual_price": 500_000},
    {"listing_id": "pg-18-bedok-south", "block": "18", "street_name": "BEDOK SOUTH ROAD", "town": "BEDOK", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(786), "storey_range": STOREY, "lease_commence_date": 1975, "sale_month": SALE_MONTH, "actual_price": 450_000},
    {"listing_id": "pg-235-bukit-batok-e5", "block": "235", "street_name": "BUKIT BATOK EAST AVENUE 5", "town": "BUKIT BATOK", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(743), "storey_range": STOREY, "lease_commence_date": 1984, "sale_month": SALE_MONTH, "actual_price": 425_000},
    {"listing_id": "pg-81-commonwealth", "block": "81", "street_name": "COMMONWEALTH CLOSE", "town": "QUEENSTOWN", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(614), "storey_range": STOREY, "lease_commence_date": 1964, "sale_month": SALE_MONTH, "actual_price": 350_000},
    {"listing_id": "pg-702-west-coast", "block": "702", "street_name": "WEST COAST ROAD", "town": "CLEMENTI", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1979, "sale_month": SALE_MONTH, "actual_price": 368_000},
    {"listing_id": "pg-643-amk-av5", "block": "643", "street_name": "ANG MO KIO AVENUE 5", "town": "ANG MO KIO", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(807), "storey_range": STOREY, "lease_commence_date": 1980, "sale_month": SALE_MONTH, "actual_price": 440_000},
    {"listing_id": "pg-24-hougang-av3", "block": "24", "street_name": "HOUGANG AVENUE 3", "town": "HOUGANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1977, "sale_month": SALE_MONTH, "actual_price": 400_000},
    {"listing_id": "pg-93-whampoa", "block": "93", "street_name": "WHAMPOA DRIVE", "town": "KALLANG/WHAMPOA", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(700), "storey_range": STOREY, "lease_commence_date": 1973, "sale_month": SALE_MONTH, "actual_price": 399_000},
    {"listing_id": "pg-629-hougang-av8", "block": "629", "street_name": "HOUGANG AVENUE 8", "town": "HOUGANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(689), "storey_range": STOREY, "lease_commence_date": 1986, "sale_month": SALE_MONTH, "actual_price": 430_000},
    {"listing_id": "pg-504-amk-av8", "block": "504", "street_name": "ANG MO KIO AVENUE 8", "town": "ANG MO KIO", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(731), "storey_range": STOREY, "lease_commence_date": 1979, "sale_month": SALE_MONTH, "actual_price": 488_888},
    {"listing_id": "pg-3-st-george", "block": "3", "street_name": "SAINT GEORGE'S ROAD", "town": "KALLANG/WHAMPOA", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(635), "storey_range": STOREY, "lease_commence_date": 1975, "sale_month": SALE_MONTH, "actual_price": 400_000},
    {"listing_id": "pg-333-amk-av1", "block": "333", "street_name": "ANG MO KIO AVENUE 1", "town": "ANG MO KIO", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1980, "sale_month": SALE_MONTH, "actual_price": 429_000},
    {"listing_id": "pg-5-ghim-moh", "block": "5", "street_name": "GHIM MOH ROAD", "town": "QUEENSTOWN", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(700), "storey_range": STOREY, "lease_commence_date": 1975, "sale_month": SALE_MONTH, "actual_price": 428_000},
    {"listing_id": "pg-104-potong-pasir", "block": "104", "street_name": "POTONG PASIR AVENUE 1", "town": "TOA PAYOH", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(797), "storey_range": STOREY, "lease_commence_date": 1984, "sale_month": SALE_MONTH, "actual_price": 618_000},
    {"listing_id": "pg-52-teban-gardens", "block": "52", "street_name": "TEBAN GARDENS ROAD", "town": "JURONG EAST", "flat_type": "4 ROOM", "floor_area_sqm": sqft_to_sqm(893), "storey_range": STOREY, "lease_commence_date": 1985, "sale_month": SALE_MONTH, "actual_price": 479_999},
    {"listing_id": "pg-18c-circuit", "block": "18C", "street_name": "CIRCUIT ROAD", "town": "GEYLANG", "flat_type": "4 ROOM", "floor_area_sqm": sqft_to_sqm(1001), "storey_range": STOREY, "lease_commence_date": 2015, "sale_month": SALE_MONTH, "actual_price": 979_000},
    {"listing_id": "pg-14-dover-close-east", "block": "14", "street_name": "DOVER CLOSE EAST", "town": "QUEENSTOWN", "flat_type": "5 ROOM", "floor_area_sqm": sqft_to_sqm(1281), "storey_range": STOREY, "lease_commence_date": 1977, "sale_month": SALE_MONTH, "actual_price": 938_000},
]

bulk_df = predict_bulk_listings(listings, bundle=bundle)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 30)
print("Full result columns:", list(bulk_df.columns))
print()

if "actual_price" in bulk_df.columns:
    eval_each = per_listing_evaluation_table(bulk_df)
    print("--- Per-listing evaluation (asking price vs model) ---")
    eval_disp = eval_each.copy()
    for col in ["actual_sgd", "predicted_sgd", "abs_error_sgd"]:
        eval_disp[col] = eval_disp[col].map(lambda x: f"{x:,.2f}")
    for col in ["mape_pct", "bias_pct"]:
        eval_disp[col] = eval_disp[col].map(lambda x: f"{x:.4f}")
    print(eval_disp.to_string(index=False))
    print("\n--- Aggregate over all rows with actual_price ---")
    print(bulk_evaluation_summary(bulk_df.dropna(subset=["actual_price"])))
else:
    print(bulk_df.to_string(index=False))




Full result columns: ['block', 'street_name', 'town', 'flat_type', 'floor_area_sqm', 'storey_range', 'lease_commence_date', 'sale_month', 'predicted_resale_price', 'lookup_matched', 'matched_address_key', 'imputation_note', 'listing_id', 'actual_price', 'abs_error_sgd', 'pct_error_vs_actual', 'mape_row_pct']

--- Per-listing evaluation (asking price vs model) ---
            listing_id                 address_short actual_sgd predicted_sgd abs_error_sgd mape_pct bias_pct
pg-101-jurong-east-s13     101 JURONG EAST STREET 13 530,000.00    420,151.69    109,848.31  20.7261 -20.7261
pg-244-jurong-east-s24     244 JURONG EAST STREET 24 380,000.00    417,729.58     37,729.58   9.9288   9.9288
      pg-93-paya-lebar             93 PAYA LEBAR WAY 300,000.00    338,793.80     38,793.80  12.9313  12.9313
pg-241-jurong-east-s24     241 JURONG EAST STREET 24 468,000.00    407,151.62     60,848.38  13.0018 -13.0018
  pg-311c-clementi-av4        311C CLEMENTI AVENUE 4 675,000.00    653,210.28     21